## Polymarket Raw Data - Goldsky 

In [ ]:
import requests
import pandas as pd
import time

In [ ]:
# Estbalishing the base URL to make the connection 

BASE_URL = "https://gamma-api.polymarket.com"

slug = "fed-decision-in-april" # Change the slug to the event you want to extract data for

# request event info
response = requests.get(f"{BASE_URL}/events", params={"slug": slug})
event_data = response.json()
event_id = event_data[0]["id"]
event_markets = event_data[0]["markets"]

for market in event_markets:
    market_id = market["id"]
    clob_id = market["clobTokenIds"]
    question = market["question"]
    print("Market ID:", market_id, "Clob ID:", clob_id, "Question:", question)

Market ID: 669660 Clob ID: ["18690049947242812495755151360212639738977254879109748949267393375856311641700", "44217754360633979680316989769899520588351517398269898942558308875588541304698"] Question: Will the Fed decrease interest rates by 50+ bps after the April 2026 meeting?
Market ID: 669661 Clob ID: ["83479140651306794046790588004449066364152228067472874205697111967337978544729", "86189905157402748775272356734086074576046362545577080735663677743589826713172"] Question: Will the Fed decrease interest rates by 25 bps after the April 2026 meeting?
Market ID: 669662 Clob ID: ["63586620628756015058616403521099137018911742768824051367331188904593189743777", "31766935524058663070405983804663917978960087327501512757848582448134393241922"] Question: Will there be no change in Fed interest rates after the April 2026 meeting?
Market ID: 669663 Clob ID: ["9556122149160720922715284597610520228366807023831966638741974320131898296289", "306953561848766593738036091070504058127380151246809275123159

In [2]:
# Spicify the CLOB_YES and CLOB_NO tokens from above depending on the event you are interested in 

CLOB_YES = "63586620628756015058616403521099137018911742768824051367331188904593189743777"
CLOB_NO  = "31766935524058663070405983804663917978960087327501512757848582448134393241922"

# Connecting the goldsky api to extract data fram the orderbook subgraph 
GOLDSKY_URL = "https://api.goldsky.com/api/public/project_cl6mb8i9h0003e201j6li0diw/subgraphs/orderbook-subgraph/0.0.1/gn"

def fetch_by_field(token_id, field):
    all_records = []
    last_id = ""
    retries = 0

    while True:
        query = """
        {
          orderFilledEvents(
            first: 500
            orderBy: id
            orderDirection: asc
            where: { %s: "%s", id_gt: "%s" }
          ) {
            id
            transactionHash
            timestamp
            maker
            taker
            makerAssetId
            takerAssetId
            makerAmountFilled
            takerAmountFilled
            fee
          }
        }
        """ % (field, token_id, last_id)

        resp   = requests.post(GOLDSKY_URL, json={"query": query}, timeout=60)
        result = resp.json()

        if "errors" in result:
            retries += 1
            print(f"  Timeout on page starting after {last_id[:20]}... — retry {retries}/5")
            if retries >= 5:
                print("  Too many retries, stopping here.")
                break
            time.sleep(5 * retries)  
            continue                  # retry the same page

        retries = 0   # reset retry counter on success
        batch = result.get("data", {}).get("orderFilledEvents", [])
        if not batch:
            break

        all_records.extend(batch)
        print(f"  [{field}={token_id[:8]}...] fetched {len(batch)} | total: {len(all_records)}")

        if len(batch) < 500:
            break

        last_id = batch[-1]["id"]
        time.sleep(0.3)   # Pause between pages to avoid overloading server

    return all_records

def fetch_all_trades_for_token(token_id, label):
    print(f"\nFetching {label} trades (as makerAssetId)...")
    as_maker = fetch_by_field(token_id, "makerAssetId")

    print(f"Fetching {label} trades (as takerAssetId)...")
    as_taker = fetch_by_field(token_id, "takerAssetId")

    seen = set()
    combined = []
    for record in as_maker + as_taker:
        if record["transactionHash"] not in seen:
            seen.add(record["transactionHash"])
            combined.append(record)

    print(f"[{label}] Total unique trades: {len(combined)}")
    return combined

def to_df(raw, outcome_label):
    if not raw:
        return pd.DataFrame()
    df = pd.DataFrame(raw)
    df["outcome"]  = outcome_label
    df["datetime"] = pd.to_datetime(df["timestamp"].astype(int), unit="s", utc=True)
    df["makerAmountFilled"] = pd.to_numeric(df["makerAmountFilled"]) / 1e6
    df["takerAmountFilled"] = pd.to_numeric(df["takerAmountFilled"]) / 1e6
    df["fee"]               = pd.to_numeric(df["fee"]) / 1e6
    df["side"]         = df["makerAssetId"].apply(lambda x: "BUY" if x == "0" else "SELL")
    df["usdc_amount"]  = df.apply(lambda r: r["makerAmountFilled"] if r["side"] == "BUY" else r["takerAmountFilled"], axis=1)
    df["token_amount"] = df.apply(lambda r: r["takerAmountFilled"] if r["side"] == "BUY" else r["makerAmountFilled"], axis=1)
    df["price"]        = df["usdc_amount"] / df["token_amount"].replace(0, float("nan"))
    return df

yes_raw = fetch_all_trades_for_token(CLOB_YES, "YES")
no_raw  = fetch_all_trades_for_token(CLOB_NO,  "NO")

df_yes = to_df(yes_raw, "YES")
df_no  = to_df(no_raw,  "NO")

df = (pd.concat([df_yes, df_no], ignore_index=True)
        .sort_values("datetime")
        .reset_index(drop=True))

print(f"\nFinal DataFrame: {len(df)} rows")
print(df[["datetime", "outcome", "side", "price", "usdc_amount", "token_amount"]].head(10))


Fetching YES trades (as makerAssetId)...
  [makerAssetId=63586620...] fetched 500 | total: 500
  [makerAssetId=63586620...] fetched 500 | total: 1000
  [makerAssetId=63586620...] fetched 500 | total: 1500
  [makerAssetId=63586620...] fetched 500 | total: 2000
  [makerAssetId=63586620...] fetched 500 | total: 2500
  [makerAssetId=63586620...] fetched 500 | total: 3000
  [makerAssetId=63586620...] fetched 500 | total: 3500
  [makerAssetId=63586620...] fetched 500 | total: 4000
  [makerAssetId=63586620...] fetched 500 | total: 4500
  [makerAssetId=63586620...] fetched 500 | total: 5000
  [makerAssetId=63586620...] fetched 500 | total: 5500
  [makerAssetId=63586620...] fetched 500 | total: 6000
  [makerAssetId=63586620...] fetched 500 | total: 6500
  [makerAssetId=63586620...] fetched 500 | total: 7000
  [makerAssetId=63586620...] fetched 500 | total: 7500
  [makerAssetId=63586620...] fetched 500 | total: 8000
  [makerAssetId=63586620...] fetched 500 | total: 8500
  [makerAssetId=63586620

In [3]:
# Removing any duplicates and sorting the rows by chronologically 
df_unique = (df.sort_values("outcome", ascending=False)  
               .drop_duplicates(subset="transactionHash", keep="first")
               .sort_values("datetime")
               .reset_index(drop=True))

print(f"Before dedup: {len(df)} rows")
print(f"After dedup:  {len(df_unique)} rows")

# Calculating yes_price and no_price 
def add_prices(row):
    if row["outcome"] == "YES":
        return row["price"], round(1 - row["price"], 6)
    else:
        return round(1 - row["price"], 6), row["price"]

df_unique[["yes_price", "no_price"]] = df_unique.apply(
    add_prices, axis=1, result_type="expand"
)

Before dedup: 113276 rows
After dedup:  72391 rows


In [ ]:
# Viewing the dataframe 
df_unique

,id,transactionHash,timestamp,maker,taker,makerAssetId,takerAssetId,makerAmountFilled,takerAmountFilled,fee,outcome,datetime,side,usdc_amount,token_amount,price,yes_price,no_price
0,0x472f9219e81c8b0b39003e066195effaecba668916eb...,0x472f9219e81c8b0b39003e066195effaecba668916eb...,1763045448,0x03c71de512897f6a613979929e0f6f511b196c86,0xd218e474776403a330142299f7796e8ba32eb5c9,0,6358662062875601505861640352109913701891174276...,6.666000,11.110,0.0,YES,2025-11-13 14:50:48+00:00,BUY,6.666000,11.110,0.600,0.600,0.400
1,0xc02d3aee7013ec94656ada3fbf901d00d279d9c15fb4...,0xc02d3aee7013ec94656ada3fbf901d00d279d9c15fb4...,1763046452,0x03c71de512897f6a613979929e0f6f511b196c86,0x63d43bbb87f85af03b8f2f9e2fad7b54334fa2f1,0,6358662062875601505861640352109913701891174276...,23.334000,38.890,0.0,YES,2025-11-13 15:07:32+00:00,BUY,23.334000,38.890,0.600,0.600,0.400
2,0xc06ddd9d33878964c126303180adfa1561c71533a3ea...,0xc06ddd9d33878964c126303180adfa1561c71533a3ea...,1763046452,0x03c71de512897f6a613979929e0f6f511b196c86,0xd218e474776403a330142299f7796e8ba32eb5c9,0,6358662062875601505861640352109913701891174276...,23.712000,39.520,0.0,YES,2025-11-13 15:07:32+00:00,BUY,23.712000,39.520,0.600,0.600,0.400
3,0xae61cd977826bcc92050c701000aa8f2df172dcde689...,0xae61cd977826bcc92050c701000aa8f2df172dcde689...,1763049920,0x03c71de512897f6a613979929e0f6f511b196c86,0x63d43bbb87f85af03b8f2f9e2fad7b54334fa2f1,0,6358662062875601505861640352109913701891174276...,6.000000,10.000,0.0,YES,2025-11-13 16:05:20+00:00,BUY,6.000000,10.000,0.600,0.600,0.400
4,0x14a1e26bccd8f58dd092dae86ad436a358547f59b5fa...,0x14a1e26bccd8f58dd092dae86ad436a358547f59b5fa...,1763069884,0x03c71de512897f6a613979929e0f6f511b196c86,0x63d43bbb87f85af03b8f2f9e2fad7b54334fa2f1,0,6358662062875601505861640352109913701891174276...,0.288000,0.480,0.0,YES,2025-11-13 21:38:04+00:00,BUY,0.288000,0.480,0.600,0.600,0.400
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72386,0x63e70a0093e56cb5f206d1b703a759c0883c46ed10c4...,0x63e70a0093e56cb5f206d1b703a759c0883c46ed10c4...,1777373858,0x9873ebe2fd2312adb5274af3400bc016ee5b92c5,0xc5d563a36ae78145c45a50134d48a1215220f80a,0,6358662062875601505861640352109913701891174276...,168.999831,169.169,0.0,YES,2026-04-28 10:57:38+00:00,BUY,168.999831,169.169,0.999,0.999,0.001
72387,0x093ac72a7d050218e61b74eb397d7d49b1f02bd5f7c0...,0x093ac72a7d050218e61b74eb397d7d49b1f02bd5f7c0...,1777373864,0x76cc009376083c69e860f8573abcac98f22eb82f,0xc5d563a36ae78145c45a50134d48a1215220f80a,0,6358662062875601505861640352109913701891174276...,88.699212,88.788,0.0,YES,2026-04-28 10:57:44+00:00,BUY,88.699212,88.788,0.999,0.999,0.001
72388,0xc8a08bfd574e29786c7335c51a1b892fb575d822568d...,0xc8a08bfd574e29786c7335c51a1b892fb575d822568d...,1777373914,0x1b306f18f10444976c608db2bb4496afb78fff28,0xc5d563a36ae78145c45a50134d48a1215220f80a,0,6358662062875601505861640352109913701891174276...,94.999905,95.095,0.0,YES,2026-04-28 10:58:34+00:00,BUY,94.999905,95.095,0.999,0.999,0.001
72389,0xf2024f35ea85530cb7d1b55cad414d2e4bc070383402...,0xf2024f35ea85530cb7d1b55cad414d2e4bc070383402...,1777374008,0x34740a48fd8c16df97d69a7f645c7dbcb8f98535,0xc5d563a36ae78145c45a50134d48a1215220f80a,0,6358662062875601505861640352109913701891174276...,169.999830,170.170,0.0,YES,2026-04-28 11:00:08+00:00,BUY,169.999830,170.170,0.999,0.999,0.001


In [ ]:
# Saving the dataframe in the defined directory 
# Ensure that you pick the directory corresponding to the event's data you have extracted in this example for 0bp cut 
df_unique.to_csv("../../../data/raw/Polymarket/data_0bp_cut/test_PM_APR_2026.csv", index=False)